# NB33 — KANSER Panel: Gelismis Feature Engineering + Ablasyon

**Tarih:** 2026-06-23
**Baseline:** NB32 Boot-F1=0.7300 (P9_REVERSE_6040 + CatBoost + 4-FE)
**Hedef:** Literatur destekli 21-feature FE ile baseline'i gecmek

In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json, gc
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_score, recall_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier

import lightgbm as lgb
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False
    print('[UYARI] catboost bulunamadi')

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'teknofest_model'))
sys.path.insert(0, os.path.abspath('..'))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

# --- Sabitler ---
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
N_OOF_FOLDS = 5
BOOT_SEED = 123
PANEL_SPLIT_FRAC = 0.50

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'v18_advanced_fe')
os.makedirs(RESULTS_DIR, exist_ok=True)
REPORTS_DIR = os.path.join(PROJECT_ROOT, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f'SEED={SEED}, PROJECT_ROOT={PROJECT_ROOT}')
print(f'Results -> {RESULTS_DIR}')

SEED=42, PROJECT_ROOT=/Users/tefe/teknofest_model/teknofest_model
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL

data_dir = os.path.join(PROJECT_ROOT, 'data', 'real_data')
df_master = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_MASTER.csv'))
df_kanser = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_KANSER.csv'))
df_cftr = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_CFTR.csv'))
df_pah = pd.read_csv(os.path.join(data_dir, 'YARISMA_TRAIN_PAH.csv'))

print(f'MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})')
print(f'KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})')
print(f'CFTR:   {df_cftr.shape} (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})')
print(f'PAH:    {df_pah.shape} (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})')

# COMBINED: MASTER+PAH+CFTR (KANSER HARIC)
df_combined_raw = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f'\nCOMBINED: {df_combined_raw.shape} (pos={df_combined_raw[TARGET].sum()}, neg={(df_combined_raw[TARGET]==0).sum()})')

# Cross-panel birebir-ayni satir drop
feat_cols_raw = [c for c in df_kanser.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_row = panel_df.loc[panel_df[ID_COL] == vid, check_cols].iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_kanser, df_master, feat_cols_raw, TARGET)
if dup_ids:
    df_kanser = df_kanser[~df_kanser[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f'KANSER: {len(dup_ids)} birebir-ayni satir drop -> {df_kanser.shape}')
else:
    print('KANSER: birebir-ayni satir yok')

# Sutun temizligi
constant_cols = [c for c in feat_cols_raw if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, feat_cols_raw)
drop_cols = set(constant_cols) | dup_drop
keep_cols = [c for c in feat_cols_raw if c not in drop_cols]
print(f'Constant: {len(constant_cols)}, Dup pairs: {len(dup_pairs)} -> drop {len(dup_drop)}')
print(f'Toplam drop: {len(drop_cols)}, Kalan feature: {len(keep_cols)}')

for _df in [df_master, df_kanser, df_cftr, df_pah]:
    for c in drop_cols:
        if c in _df.columns:
            _df.drop(columns=[c], inplace=True)

df_combined = pd.concat([df_master, df_pah, df_cftr], ignore_index=True)
print(f'\nFinal: MASTER={df_master.shape}, COMBINED={df_combined.shape}, KANSER={df_kanser.shape}')

# KANSER 50/50 split (NB31 ile ayni)
def panel_5050_split(df):
    pos = df[df[TARGET]==1].sample(frac=1.0, random_state=SEED)
    neg = df[df[TARGET]==0].sample(frac=1.0, random_state=SEED)
    npos = int(round(len(pos) * PANEL_SPLIT_FRAC))
    nneg = int(round(len(neg) * PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:npos], neg.iloc[:nneg]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    te = pd.concat([pos.iloc[npos:], neg.iloc[nneg:]]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return tr, te

kanser_train, kanser_test = panel_5050_split(df_kanser)
y_kanser_train = kanser_train[TARGET].values
y_kanser_test = kanser_test[TARGET].values
print(f'\nKANSER train: {kanser_train.shape} (pos={y_kanser_train.sum()}, neg={(y_kanser_train==0).sum()})')
print(f'KANSER test:  {kanser_test.shape} (pos={y_kanser_test.sum()}, neg={(y_kanser_test==0).sum()})')
assert y_kanser_test.sum() > 0 and (y_kanser_test==0).sum() > 0, 'Split hatasi!'

NUM_COLS_BASE = [c for c in keep_cols if df_master[c].dtype != 'object']
CAT_COLS_BASE = [c for c in keep_cols if df_master[c].dtype == 'object']
print(f'Numeric: {len(NUM_COLS_BASE)}, Categorical: {len(CAT_COLS_BASE)}')

MASTER: (2931, 353) (pos=2149, neg=782)
KANSER: (388, 353) (pos=268, neg=120)
CFTR:   (111, 353) (pos=90, neg=21)
PAH:    (372, 353) (pos=310, neg=62)

COMBINED: (3414, 353) (pos=2549, neg=865)
KANSER: 3 birebir-ayni satir drop -> (385, 353)
Constant: 0, Dup pairs: 58 -> drop 58
Toplam drop: 58, Kalan feature: 293

Final: MASTER=(2931, 295), COMBINED=(3414, 295), KANSER=(385, 295)

KANSER train: (192, 295) (pos=132, neg=60)
KANSER test:  (193, 295) (pos=133, neg=60)
Numeric: 286, Categorical: 7


In [3]:
# Cell 3: M3 Preprocessing
AA_UNK = CR.AA_UNKNOWN_TOKEN
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")
HIGH_MISS_THR = 0.50

def fit_preprocessor(train_df, keep_cols=keep_cols, target=TARGET):
    """Train uzerinde fit: median, label encoder, high-missing tespiti."""
    X = train_df[keep_cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > HIGH_MISS_THR].index.tolist()
    medians = X[num_cols].median()
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps, "keep_cols": keep_cols
    }

def transform_X(df, prep):
    """Preprocessor uygula, is_missing flagleri ekle."""
    kc = prep["keep_cols"]
    X = df[kc].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return X

print("M3 Preprocessing hazir.")

M3 Preprocessing hazir.


In [4]:
# Cell 4: Degerlendirme Altyapisi

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y); prob = np.asarray(prob)
    neg = np.where(y == 0)[0]; pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"mean": float(f1s.mean()), "std": float(f1s.std()),
            "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))}

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def train_metrics_at(y_train, p_train, thr):
    yp = (p_train >= thr).astype(int)
    return {
        "train_f1": float(_f1_pos(y_train, yp)),
        "train_mcc": float(matthews_corrcoef(y_train, yp)),
        "train_prec": float(precision_score(y_train, yp, pos_label=1, zero_division=0)),
        "train_rec": float(recall_score(y_train, yp, pos_label=1, zero_division=0))
    }

def eval_model(label, y_test, p_test, y_train, p_train):
    thr = select_threshold_8020_robust(y_train, p_train)
    boot = bootstrap_8020(y_test, p_test, thr)
    y_pred = (p_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0,1]).ravel()
    auprc = average_precision_score(y_test, p_test) if len(np.unique(y_test)) > 1 else 0.0
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    tr_met = train_metrics_at(y_train, p_train, thr)
    return {
        "boot_f1": boot["mean"], "boot_std": boot["std"],
        "boot_lo": boot["lo"], "boot_hi": boot["hi"],
        "auprc": auprc, "precision": prec, "recall": rec, "mcc": mcc,
        "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
        "thr": thr, **tr_met
    }

print("Degerlendirme altyapisi hazir.")

Degerlendirme altyapisi hazir.


In [5]:
# Cell 5: Tree Model Helpers

LGBM_PARAMS = {
    "n_estimators": 300, "num_leaves": 31, "learning_rate": 0.05,
    "min_child_samples": 20, "subsample": 0.8, "colsample_bytree": 0.8,
    "class_weight": "balanced",
    "random_state": SEED, "verbose": -1, "n_jobs": -1, "importance_type": "gain"
}
XGB_PARAMS = {
    "n_estimators": 300, "max_depth": 6, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "random_state": SEED, "verbosity": 0, "n_jobs": -1
}
CB_PARAMS = {
    "iterations": 300, "depth": 6, "learning_rate": 0.05,
    "auto_class_weights": "Balanced",
    "random_seed": SEED, "verbose": 0
}

def _prep_tree(X_df):
    """Kategorikleri lgbm/xgb icin hazirla."""
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    Xn = X_df.copy()
    le_maps = {}
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
        le_maps[c] = le
    return Xn, cat_cols, le_maps

def _apply_le(X_df, cat_cols, le_maps):
    Xn = X_df.copy()
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = le_maps[c]
        Xn[c] = Xn[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return Xn

def oof_tree(model_type, X_train_df, y_train, X_test_df, n_splits=N_OOF_FOLDS):
    """Tree model OOF: train uzerinde oof + test uzerinde full-model proba."""
    X_tr, cat_cols, le_maps = _prep_tree(X_train_df)
    X_te = _apply_le(X_test_df, cat_cols, le_maps)

    # lgbm icin category type
    if model_type == "lgbm":
        for c in cat_cols:
            X_tr[c] = X_tr[c].astype("category")
            X_te[c] = pd.Categorical(X_te[c], categories=X_tr[c].cat.categories)

    oof = np.zeros(len(y_train))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for tri, vai in skf.split(X_tr, y_train):
        if model_type == "lgbm":
            m = LGBMClassifier(**LGBM_PARAMS)
            m.fit(X_tr.iloc[tri], y_train[tri], categorical_feature=cat_cols)
        elif model_type == "xgb":
            spw = float((y_train[tri]==0).sum()) / max(float((y_train[tri]==1).sum()), 1)
            m = XGBClassifier(**{**XGB_PARAMS, "scale_pos_weight": spw})
            m.fit(X_tr.iloc[tri], y_train[tri])
        elif model_type == "catboost":
            cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]
            m = CatBoostClassifier(**CB_PARAMS)
            m.fit(X_tr.iloc[tri], y_train[tri], cat_features=cat_idx, silent=True)
        oof[vai] = m.predict_proba(X_tr.iloc[vai])[:, 1]

    # Full model
    if model_type == "lgbm":
        mf = LGBMClassifier(**LGBM_PARAMS)
        mf.fit(X_tr, y_train, categorical_feature=cat_cols)
    elif model_type == "xgb":
        spw = float((y_train==0).sum()) / max(float((y_train==1).sum()), 1)
        mf = XGBClassifier(**{**XGB_PARAMS, "scale_pos_weight": spw})
        mf.fit(X_tr, y_train)
    elif model_type == "catboost":
        cat_idx = [list(X_tr.columns).index(c) for c in cat_cols]
        mf = CatBoostClassifier(**CB_PARAMS)
        mf.fit(X_tr, y_train, cat_features=cat_idx, silent=True)

    test_proba = mf.predict_proba(X_te)[:, 1]
    return oof, test_proba

print("Tree model helpers hazir: lgbm, xgb, catboost")

Tree model helpers hazir: lgbm, xgb, catboost


In [6]:
# Cell 7: Feature Engineering (NB16 Cell 4 — Grantham/BLOSUM62/stopgain)

_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}

def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a))

_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)
        _B62[(row_aa, col_aa)] = v; _B62[(col_aa, row_aa)] = v

def blosum62(a, b):
    return _B62.get((a, b), 0)

# === New Physicochemical Lookup Tables ===

# Kyte-Doolittle hydrophobicity
_HYDRO = {'A':1.8,'R':-4.5,'N':-3.5,'D':-3.5,'C':2.5,'Q':-3.5,'E':-3.5,'G':-0.4,
           'H':-3.2,'I':4.5,'L':3.8,'K':-3.9,'M':1.9,'F':2.8,'P':-1.6,'S':-0.8,
           'T':-0.7,'W':-0.9,'Y':-1.3,'V':4.2}

# Zamyakin volume (A^3)
_VOLUME = {'A':88.6,'R':173.4,'N':114.1,'D':111.1,'C':108.5,'Q':143.8,'E':138.4,'G':60.1,
            'H':153.2,'I':166.7,'L':166.7,'K':168.6,'M':162.9,'F':189.9,'P':112.7,'S':89.0,
            'T':116.1,'W':227.8,'Y':193.6,'V':140.0}

# Standard molecular weight (Da)
_MW = {'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.16,'Q':146.15,'E':147.13,'G':75.03,
       'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,'P':115.13,'S':105.09,
       'T':119.12,'W':204.23,'Y':181.19,'V':117.15}

# Zimmerman polarity
_POLAR = {'A':0.0,'R':52.0,'N':3.38,'D':49.7,'C':1.48,'Q':3.53,'E':49.9,'G':0.0,
           'H':51.6,'I':0.13,'L':0.13,'K':49.5,'M':1.43,'F':0.35,'P':1.58,'S':1.67,
           'T':1.66,'W':2.10,'Y':1.61,'V':0.13}

# Charge classes for charge_change feature
_CHARGE = {'R':'+','K':'+','H':'+','D':'-','E':'-',
           'A':'0','N':'0','C':'0','Q':'0','G':'0','I':'0','L':'0',
           'M':'0','F':'0','P':'0','S':'0','T':'0','W':'0','Y':'0','V':'0'}

# === add_fe_v2 ===

def add_fe_v2(df, freq_cols=None, binary_flag_cols=None):
    out = df.copy()
    a1 = out["AA_1"].astype("object")
    a2 = out["AA_2"].astype("object")

    # === GROUP F: Existing FE (from NB32) ===
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v):
        return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    out["fe_grantham"] = out.apply(lambda r: grantham(r["AA_1"], r["AA_2"]) if (isinstance(r["AA_1"], str) and isinstance(r["AA_2"], str) and r["AA_1"] in STANDARD_AA and r["AA_2"] in STANDARD_AA) else -1, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(lambda r: blosum62(r["AA_1"], r["AA_2"]) if (isinstance(r["AA_1"], str) and isinstance(r["AA_2"], str) and r["AA_1"] in STANDARD_AA and r["AA_2"] in STANDARD_AA) else 0, axis=1).astype(float)

    # === GROUP A: FCS-style Frequency x Conservation (3 features) ===
    if freq_cols is None:
        al_cols = [c for c in out.columns if c.startswith("AL_")]
        freq_cols = []
        for c in al_cols:
            vals = out[c].dropna()
            if len(vals) == 0: continue
            uniq = vals.unique()
            if len(uniq) > 2 and vals.min() >= 0 and vals.mean() < 0.1:
                freq_cols.append(c)

    if freq_cols:
        max_pop_freq = out[freq_cols].max(axis=1)
        log_max_freq = np.log10(max_pop_freq.clip(lower=1e-10))
        log_max_freq = log_max_freq.where(max_pop_freq.notna(), np.nan)
    else:
        log_max_freq = pd.Series(np.nan, index=out.index)

    ek9 = out["EK_9"] if "EK_9" in out.columns else pd.Series(np.nan, index=out.index)
    ek7 = out["EK_7"] if "EK_7" in out.columns else pd.Series(np.nan, index=out.index)

    out["fe_fcs_ek9"] = log_max_freq * ek9
    out["fe_fcs_ek7"] = log_max_freq * ek7
    out["fe_log_max_freq"] = log_max_freq

    # === GROUP B: EK Score Combinations (5 features) ===
    ek_cols_map = {}
    for i in range(1, 10):
        cn = f"EK_{i}"
        if cn in out.columns:
            ek_cols_map[i] = out[cn]

    if all(k in ek_cols_map for k in [7, 9, 2]):
        out["fe_ek_mean_top3"] = pd.concat([ek_cols_map[7], ek_cols_map[9], ek_cols_map[2]], axis=1).mean(axis=1, skipna=True)
    else:
        out["fe_ek_mean_top3"] = np.nan

    if all(k in ek_cols_map for k in [7, 9, 2, 6]):
        four = pd.concat([ek_cols_map[7], ek_cols_map[9], ek_cols_map[2], ek_cols_map[6]], axis=1)
        out["fe_ek_max"] = four.max(axis=1, skipna=True)
        out["fe_ek_range"] = four.max(axis=1, skipna=True) - four.min(axis=1, skipna=True)
    else:
        out["fe_ek_max"] = np.nan
        out["fe_ek_range"] = np.nan

    if all(k in ek_cols_map for k in [4, 5, 6]):
        out["fe_ek_consensus"] = ek_cols_map[4] + ek_cols_map[5] + ek_cols_map[6]
    else:
        out["fe_ek_consensus"] = np.nan

    if all(k in ek_cols_map for k in [1, 2]):
        out["fe_ek_delta_12"] = ek_cols_map[1] - ek_cols_map[2]
    else:
        out["fe_ek_delta_12"] = np.nan

    # === GROUP C: Missingness Features (3 features) ===
    if binary_flag_cols is None:
        al_cols = [c for c in out.columns if c.startswith("AL_")]
        binary_flag_cols = []
        for c in al_cols:
            vals = out[c].dropna()
            if len(vals) == 0: continue
            if set(vals.unique()).issubset({0, 1, 0.0, 1.0}):
                binary_flag_cols.append(c)

    if freq_cols:
        out["fe_n_pop_observed"] = out[freq_cols].notna().sum(axis=1)
    else:
        out["fe_n_pop_observed"] = 0

    out["fe_n_missing_total"] = out.isnull().sum(axis=1)

    if binary_flag_cols:
        out["fe_n_binary_flags_1"] = out[binary_flag_cols].sum(axis=1, skipna=True)
    else:
        out["fe_n_binary_flags_1"] = 0

    # === GROUP D: AA Physicochemical Deltas (4 features) ===
    def _aa_delta_abs(a1_col, a2_col, lookup):
        def _calc(row):
            x, y = row[a1_col], row[a2_col]
            if isinstance(x, str) and isinstance(y, str) and x in lookup and y in lookup:
                return abs(lookup[x] - lookup[y])
            return np.nan
        return out.apply(_calc, axis=1)

    out["fe_hydro_abs"] = _aa_delta_abs("AA_1", "AA_2", _HYDRO)
    out["fe_vol_abs"] = _aa_delta_abs("AA_1", "AA_2", _VOLUME)
    out["fe_mw_abs"] = _aa_delta_abs("AA_1", "AA_2", _MW)
    out["fe_polar_abs"] = _aa_delta_abs("AA_1", "AA_2", _POLAR)

    # === GROUP E: AA Class Transitions (2 features) ===
    def _grantham_cat(val):
        if val < 0 or np.isnan(val): return np.nan
        if val == 0: return 0
        if val <= 50: return 1
        if val <= 100: return 2
        if val <= 150: return 3
        return 4
    out["fe_grantham_cat"] = out["fe_grantham"].apply(_grantham_cat)

    def _charge_change(row):
        x, y = row["AA_1"], row["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in _CHARGE and y in _CHARGE:
            return 0 if _CHARGE[x] == _CHARGE[y] else 1
        return np.nan
    out["fe_charge_change"] = out.apply(_charge_change, axis=1)

    return out

# === Feature group definitions ===
FE_GROUP_F = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]
FE_GROUP_A = ["fe_fcs_ek9", "fe_fcs_ek7", "fe_log_max_freq"]
FE_GROUP_B = ["fe_ek_mean_top3", "fe_ek_max", "fe_ek_consensus", "fe_ek_range", "fe_ek_delta_12"]
FE_GROUP_C = ["fe_n_pop_observed", "fe_n_missing_total", "fe_n_binary_flags_1"]
FE_GROUP_D = ["fe_hydro_abs", "fe_vol_abs", "fe_mw_abs", "fe_polar_abs"]
FE_GROUP_E = ["fe_grantham_cat", "fe_charge_change"]
FE_ALL = FE_GROUP_F + FE_GROUP_A + FE_GROUP_B + FE_GROUP_C + FE_GROUP_D + FE_GROUP_E

# For auto-detection consistency: detect once on training data, reuse
_det_freq = None
_det_binary = None

def detect_col_groups(df):
    """Detect freq and binary flag columns from training data. Call once, reuse."""
    global _det_freq, _det_binary
    al_cols = [c for c in df.columns if c.startswith("AL_")]
    freq_cols = []
    binary_cols = []
    for c in al_cols:
        vals = df[c].dropna()
        if len(vals) == 0: continue
        uniq = set(vals.unique())
        if uniq.issubset({0, 1, 0.0, 1.0}):
            binary_cols.append(c)
        elif len(uniq) > 2 and vals.min() >= 0 and vals.mean() < 0.1:
            freq_cols.append(c)
    _det_freq = freq_cols
    _det_binary = binary_cols
    print(f"Detected: {len(freq_cols)} freq cols, {len(binary_cols)} binary flag cols")
    return freq_cols, binary_cols

print(f"add_fe_v2 hazir. Feature gruplari: F={len(FE_GROUP_F)}, A={len(FE_GROUP_A)}, B={len(FE_GROUP_B)}, C={len(FE_GROUP_C)}, D={len(FE_GROUP_D)}, E={len(FE_GROUP_E)}, Toplam={len(FE_ALL)}")

add_fe_v2 hazir. Feature gruplari: F=4, A=3, B=5, C=3, D=4, E=2, Toplam=21


In [7]:
# Cell 7: Training Pool (P9_REVERSE_6040 -- NB32 champion)

def build_pool():
    """P9_REVERSE_6040: COMBINED %60B/%40P."""
    comb_neg = df_combined[df_combined[TARGET]==0]
    comb_pos = df_combined[df_combined[TARGET]==1]
    n_neg = len(comb_neg)
    n_pos = max(1, int(round(n_neg * 0.40 / 0.60)))
    pos_sample = comb_pos.sample(n=min(n_pos, len(comb_pos)), random_state=SEED)
    pool = pd.concat([comb_neg, pos_sample]).sample(
        frac=1.0, random_state=SEED).reset_index(drop=True)
    return pool

pool_df = build_pool()
print(f"Pool P9_REVERSE_6040: {pool_df.shape} (pos={pool_df[TARGET].sum()}, neg={(pool_df[TARGET]==0).sum()})")

# Detect column groups on pool (training data)
freq_cols, binary_flag_cols = detect_col_groups(pool_df)

Pool P9_REVERSE_6040: (1442, 295) (pos=577, neg=865)
Detected: 119 freq cols, 25 binary flag cols


In [8]:
# Cell 8: Exp 1 -- Grup Ablasyonu
print("="*70)
print("[Exp 1] Feature Group Ablation")
print("="*70)

all_results = {}
all_test_probas = {}

ablation_configs = {
    "E1a_GroupA_FCS":      FE_GROUP_F + FE_GROUP_A,
    "E1b_GroupB_EKCombo":  FE_GROUP_F + FE_GROUP_B,
    "E1c_GroupC_Missing":  FE_GROUP_F + FE_GROUP_C,
    "E1d_GroupDE_AAdelta": FE_GROUP_F + FE_GROUP_D + FE_GROUP_E,
    "E1e_All_ABCDEF":     FE_ALL,
    "E1f_Baseline_F":     FE_GROUP_F,
    "E1g_NoFE":           [],
    "E1h_All_minus_C":    FE_GROUP_F + FE_GROUP_A + FE_GROUP_B + FE_GROUP_D + FE_GROUP_E,
}

for exp_name, fe_cols in ablation_configs.items():
    print(f"\n--- {exp_name} ({len(fe_cols)} FE features) ---")
    try:
        # Apply FE
        if fe_cols:
            pool_fe = add_fe_v2(pool_df, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
            test_fe = add_fe_v2(kanser_test, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
            train_fe = add_fe_v2(kanser_train, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
            # Only keep specified FE cols
            fe_keep = keep_cols + [c for c in fe_cols if c in pool_fe.columns]
        else:
            pool_fe = pool_df
            test_fe = kanser_test
            train_fe = kanser_train
            fe_keep = keep_cols

        pp = fit_preprocessor(pool_fe, keep_cols=fe_keep)
        X_pool = transform_X(pool_fe, pp)
        X_test = transform_X(test_fe, pp)
        y_pool = pool_fe[TARGET].values

        oof, test_proba = oof_tree("catboost", X_pool, y_pool, X_test)

        res = eval_model(exp_name, y_kanser_test, test_proba, y_pool, oof)
        all_results[exp_name] = res
        all_test_probas[exp_name] = test_proba

        print(f"  Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  "
              f"Prec={res['precision']:.3f}  FP={res['fp']}  FN={res['fn']}  "
              f"Train-F1={res['train_f1']:.3f}  Gap={res['train_f1']-res['boot_f1']:.3f}")
    except Exception as e:
        print(f"  HATA: {e}")
        import traceback; traceback.print_exc()
        all_results[exp_name] = None

# Summary table
print("\n" + "="*70)
print("ABLASYON OZET")
print("="*70)
print(f"{'Deney':<25} {'Boot-F1':>8} {'Std':>6} {'Prec':>6} {'FP':>4} {'FN':>4} {'Gap':>6}")
print("-"*70)
baseline_f1 = None
for exp_name in ablation_configs:
    r = all_results.get(exp_name)
    if r:
        if exp_name == "E1f_Baseline_F":
            baseline_f1 = r["boot_f1"]
        delta = f" ({r['boot_f1']-baseline_f1:+.4f})" if baseline_f1 is not None else ""
        print(f"{exp_name:<25} {r['boot_f1']:>8.4f} {r['boot_std']:>6.3f} "
              f"{r['precision']:>6.3f} {r['fp']:>4d} {r['fn']:>4d} "
              f"{r['train_f1']-r['boot_f1']:>6.3f}{delta}")

[Exp 1] Feature Group Ablation

--- E1a_GroupA_FCS (7 FE features) ---
  Boot-F1=0.7200 +/- 0.026  Prec=0.926  FP=10  FN=8  Train-F1=0.745  Gap=0.025

--- E1b_GroupB_EKCombo (9 FE features) ---
  Boot-F1=0.6929 +/- 0.046  Prec=0.922  FP=10  FN=14  Train-F1=0.724  Gap=0.032

--- E1c_GroupC_Missing (7 FE features) ---
  Boot-F1=0.6998 +/- 0.055  Prec=0.928  FP=9  FN=17  Train-F1=0.727  Gap=0.028

--- E1d_GroupDE_AAdelta (10 FE features) ---
  Boot-F1=0.7576 +/- 0.045  Prec=0.944  FP=7  FN=14  Train-F1=0.727  Gap=-0.031

--- E1e_All_ABCDEF (21 FE features) ---
  Boot-F1=0.6899 +/- 0.035  Prec=0.918  FP=11  FN=10  Train-F1=0.742  Gap=0.052

--- E1f_Baseline_F (4 FE features) ---
  Boot-F1=0.7300 +/- 0.033  Prec=0.933  FP=9  FN=8  Train-F1=0.734  Gap=0.004

--- E1g_NoFE (0 FE features) ---
  Boot-F1=0.7104 +/- 0.042  Prec=0.930  FP=9  FN=14  Train-F1=0.720  Gap=0.010

--- E1h_All_minus_C (18 FE features) ---
  Boot-F1=0.7410 +/- 0.049  Prec=0.944  FP=7  FN=16  Train-F1=0.714  Gap=-0.027

AB

In [9]:
# Cell 9: Exp 2 -- FE Versiyon Karsilastirmasi
print("\n" + "="*70)
print("[Exp 2] FE Versiyon Karsilastirmasi")
print("="*70)

# Also test with LGBM and XGB for robustness check
for model_type in ["catboost", "lgbm", "xgb"]:
    for fe_label, fe_cols in [("no_fe", []), ("nb32_fe", FE_GROUP_F), ("v2_full", FE_ALL), ("v2_safe", [c for c in FE_ALL if c not in FE_GROUP_C])]:
        exp_key = f"E2_{model_type}_{fe_label}"
        print(f"\n  {exp_key}:")
        try:
            if fe_cols:
                pool_fe = add_fe_v2(pool_df, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
                test_fe = add_fe_v2(kanser_test, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
                fe_keep = keep_cols + [c for c in fe_cols if c in pool_fe.columns]
            else:
                pool_fe = pool_df
                test_fe = kanser_test
                fe_keep = keep_cols

            pp = fit_preprocessor(pool_fe, keep_cols=fe_keep)
            X_pool = transform_X(pool_fe, pp)
            X_test = transform_X(test_fe, pp)
            y_pool = pool_fe[TARGET].values

            oof, test_proba = oof_tree(model_type, X_pool, y_pool, X_test)
            res = eval_model(exp_key, y_kanser_test, test_proba, y_pool, oof)
            all_results[exp_key] = res
            all_test_probas[exp_key] = test_proba

            print(f"    Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  "
                  f"Prec={res['precision']:.3f}  FP={res['fp']}  FN={res['fn']}")
        except Exception as e:
            print(f"    HATA: {e}")
            all_results[exp_key] = None

# Summary
print("\n  FE Versiyon Karsilastirma Ozet:")
for mt in ["catboost", "lgbm", "xgb"]:
    print(f"\n  {mt.upper()}:")
    for fe_label in ["no_fe", "nb32_fe", "v2_full", "v2_safe"]:
        k = f"E2_{mt}_{fe_label}"
        r = all_results.get(k)
        if r:
            print(f"    {fe_label:<12} Boot-F1={r['boot_f1']:.4f}  Prec={r['precision']:.3f}")


[Exp 2] FE Versiyon Karsilastirmasi

  E2_catboost_no_fe:
    Boot-F1=0.7104 +/- 0.042  Prec=0.930  FP=9  FN=14

  E2_catboost_nb32_fe:
    Boot-F1=0.7300 +/- 0.033  Prec=0.933  FP=9  FN=8

  E2_catboost_v2_full:
    Boot-F1=0.6899 +/- 0.035  Prec=0.918  FP=11  FN=10

  E2_catboost_v2_safe:
    Boot-F1=0.7410 +/- 0.049  Prec=0.944  FP=7  FN=16

  E2_lgbm_no_fe:
    Boot-F1=0.6601 +/- 0.057  Prec=0.918  FP=10  FN=21

  E2_lgbm_nb32_fe:
    Boot-F1=0.6578 +/- 0.058  Prec=0.917  FP=10  FN=22

  E2_lgbm_v2_full:
    Boot-F1=0.7447 +/- 0.047  Prec=0.944  FP=7  FN=16

  E2_lgbm_v2_safe:
    Boot-F1=0.7065 +/- 0.049  Prec=0.929  FP=9  FN=15

  E2_xgb_no_fe:
    Boot-F1=0.6652 +/- 0.054  Prec=0.918  FP=10  FN=21

  E2_xgb_nb32_fe:
    Boot-F1=0.6876 +/- 0.044  Prec=0.921  FP=10  FN=16

  E2_xgb_v2_full:
    Boot-F1=0.7081 +/- 0.046  Prec=0.929  FP=9  FN=15

  E2_xgb_v2_safe:
    Boot-F1=0.6923 +/- 0.056  Prec=0.932  FP=8  FN=24

  FE Versiyon Karsilastirma Ozet:

  CATBOOST:
    no_fe        

In [10]:
# Cell 10: Exp 3 -- RFE ile Optimal Feature Subset
print("\n" + "="*70)
print("[Exp 3] RFE - Optimal Feature Subset")
print("="*70)

# Start with best FE config from Exp 1
e1_candidates = [(k,v) for k,v in all_results.items() if v and k.startswith("E1")]
best_e1 = max(e1_candidates, key=lambda x: x[1]["boot_f1"])
print(f"Starting from best Exp 1: {best_e1[0]} (Boot-F1={best_e1[1]['boot_f1']:.4f})")

# Use the safe version (no Group C) if it performs similarly
e1e_r = all_results.get("E1e_All_ABCDEF")
e1h_r = all_results.get("E1h_All_minus_C")
if e1e_r and e1h_r and (e1e_r["boot_f1"] - e1h_r["boot_f1"]) < 0.005:
    rfe_start_cols = [c for c in FE_ALL if c not in FE_GROUP_C]
    print("Using leakage-safe (All minus C) as RFE starting point")
else:
    rfe_start_cols = FE_ALL.copy()
    print("Using full FE as RFE starting point")

# Apply FE and get feature importance
pool_fe = add_fe_v2(pool_df, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
test_fe = add_fe_v2(kanser_test, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)

current_fe = [c for c in rfe_start_cols if c in pool_fe.columns]
rfe_results = []

while len(current_fe) >= 2:
    fe_keep = keep_cols + current_fe
    pp = fit_preprocessor(pool_fe, keep_cols=fe_keep)
    X_pool = transform_X(pool_fe, pp)
    X_test = transform_X(test_fe, pp)
    y_pool = pool_fe[TARGET].values

    # Train full CatBoost for importance
    from catboost import Pool as CatPool
    X_tr_np, cat_cols_cb, le_maps_cb = _prep_tree(X_pool)
    X_te_np = _apply_le(X_test, cat_cols_cb, le_maps_cb)
    cat_idx = [list(X_tr_np.columns).index(c) for c in cat_cols_cb]

    cb = CatBoostClassifier(**CB_PARAMS)
    cb.fit(X_tr_np, y_pool, cat_features=cat_idx, silent=True)

    # Feature importance (only for FE cols)
    fi = pd.Series(cb.feature_importances_, index=X_tr_np.columns)
    fi_fe = fi[[c for c in current_fe if c in fi.index]].sort_values()

    # Evaluate
    oof, test_proba = oof_tree("catboost", X_pool, y_pool, X_test)
    res = eval_model(f"RFE_{len(current_fe)}", y_kanser_test, test_proba, y_pool, oof)

    rfe_results.append({
        "n_fe": len(current_fe),
        "fe_cols": current_fe.copy(),
        "boot_f1": res["boot_f1"],
        "boot_std": res["boot_std"],
        "precision": res["precision"],
        "dropped": fi_fe.index[0] if len(fi_fe) > 0 else None,
        "dropped_importance": float(fi_fe.iloc[0]) if len(fi_fe) > 0 else None
    })

    if len(fi_fe) > 0:
        print(f"  n_fe={len(current_fe):>2}  Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}  "
              f"Drop: {fi_fe.index[0]} (imp={fi_fe.iloc[0]:.2f})")
    else:
        print(f"  n_fe={len(current_fe):>2}  Boot-F1={res['boot_f1']:.4f} +/- {res['boot_std']:.3f}")

    all_results[f"E3_RFE_{len(current_fe)}"] = res

    # Remove least important FE feature
    if len(fi_fe) > 0:
        current_fe = [c for c in current_fe if c != fi_fe.index[0]]
    else:
        break

# Find optimal
if rfe_results:
    rfe_df = pd.DataFrame(rfe_results)
    best_rfe = rfe_df.loc[rfe_df["boot_f1"].idxmax()]
    print(f"\nOptimal: {int(best_rfe['n_fe'])} FE features, Boot-F1={best_rfe['boot_f1']:.4f}")
    print(f"Features: {best_rfe['fe_cols']}")


[Exp 3] RFE - Optimal Feature Subset
Starting from best Exp 1: E1d_GroupDE_AAdelta (Boot-F1=0.7576)
Using leakage-safe (All minus C) as RFE starting point
  n_fe=18  Boot-F1=0.7410 +/- 0.049  Drop: fe_aa_stopgain (imp=0.00)
  n_fe=17  Boot-F1=0.6470 +/- 0.037  Drop: fe_aa_nonstandard (imp=0.06)
  n_fe=16  Boot-F1=0.6914 +/- 0.033  Drop: fe_grantham_cat (imp=0.14)
  n_fe=15  Boot-F1=0.6979 +/- 0.043  Drop: fe_charge_change (imp=0.25)
  n_fe=14  Boot-F1=0.7325 +/- 0.047  Drop: fe_vol_abs (imp=0.27)
  n_fe=13  Boot-F1=0.7167 +/- 0.041  Drop: fe_ek_range (imp=0.51)
  n_fe=12  Boot-F1=0.7329 +/- 0.040  Drop: fe_hydro_abs (imp=0.69)
  n_fe=11  Boot-F1=0.6839 +/- 0.036  Drop: fe_log_max_freq (imp=0.60)
  n_fe=10  Boot-F1=0.7513 +/- 0.031  Drop: fe_ek_consensus (imp=0.72)
  n_fe= 9  Boot-F1=0.7174 +/- 0.040  Drop: fe_mw_abs (imp=1.10)
  n_fe= 8  Boot-F1=0.7262 +/- 0.040  Drop: fe_ek_mean_top3 (imp=0.80)
  n_fe= 7  Boot-F1=0.6708 +/- 0.039  Drop: fe_ek_delta_12 (imp=1.17)
  n_fe= 6  Boot-F1=0.

In [11]:
# Cell 11: Sonuc Derleme + Gorsellestirmeler
print("\n" + "="*70)
print("SONUC DERLEMESI")
print("="*70)

# Compile all results
rows = []
for k, v in sorted(all_results.items()):
    if v is None: continue
    rows.append({"experiment": k, **v})
results_df = pd.DataFrame(rows).sort_values("boot_f1", ascending=False).reset_index(drop=True)

csv_path = os.path.join(RESULTS_DIR, "advanced_fe_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"Sonuclar: {csv_path}")
print(f"Toplam deney: {len(results_df)}")
print(f"\nEn iyi: {results_df.iloc[0]['experiment']} (Boot-F1={results_df.iloc[0]['boot_f1']:.4f})")
print(f"NB32 Baseline: 0.7300")

# Fig 1: Group Ablation Bar Chart
fig1, ax1 = plt.subplots(figsize=(12, 6))
abl_names = list(ablation_configs.keys())
abl_f1s = [all_results.get(n, {}).get("boot_f1", 0) for n in abl_names]
abl_stds = [all_results.get(n, {}).get("boot_std", 0) for n in abl_names]
colors = ['#2196F3' if n != "E1f_Baseline_F" else '#FF5722' for n in abl_names]
bars = ax1.bar(range(len(abl_names)), abl_f1s, yerr=abl_stds, capsize=4, color=colors, alpha=0.8)
ax1.axhline(y=0.7300, color='red', linestyle='--', linewidth=1.5, label='NB32 Baseline (0.7300)')
ax1.set_xticks(range(len(abl_names)))
ax1.set_xticklabels([n.replace("E1", "").replace("_", "\n") for n in abl_names], rotation=45, ha='right', fontsize=8)
ax1.set_ylabel("Bootstrap 80/20 F1")
ax1.set_title("Exp 1: Feature Group Ablation")
ax1.legend()
plt.tight_layout()
fig1_path = os.path.join(RESULTS_DIR, "fig1_group_ablation.png")
fig1.savefig(fig1_path, dpi=150)
plt.close(fig1)
print(f"Fig 1: {fig1_path}")

# Fig 2: FE Version Comparison (grouped bar)
fig2, ax2 = plt.subplots(figsize=(10, 6))
fe_labels = ["no_fe", "nb32_fe", "v2_full", "v2_safe"]
model_types = ["catboost", "lgbm", "xgb"]
x = np.arange(len(fe_labels))
width = 0.25
for i, mt in enumerate(model_types):
    vals = [all_results.get(f"E2_{mt}_{fl}", {}).get("boot_f1", 0) for fl in fe_labels]
    ax2.bar(x + i*width, vals, width, label=mt.upper(), alpha=0.8)
ax2.axhline(y=0.7300, color='red', linestyle='--', linewidth=1.5, label='NB32 Baseline')
ax2.set_xticks(x + width)
ax2.set_xticklabels(["No FE", "NB32 FE (4)", "v2 Full (21)", "v2 Safe (18)"])
ax2.set_ylabel("Bootstrap 80/20 F1")
ax2.set_title("Exp 2: FE Version Comparison")
ax2.legend()
plt.tight_layout()
fig2_path = os.path.join(RESULTS_DIR, "fig2_fe_comparison.png")
fig2.savefig(fig2_path, dpi=150)
plt.close(fig2)
print(f"Fig 2: {fig2_path}")

# Fig 3: Feature Importance (from best model)
fig3, ax3 = plt.subplots(figsize=(10, 8))
pool_fe = add_fe_v2(pool_df, freq_cols=freq_cols, binary_flag_cols=binary_flag_cols)
best_fe_keep = keep_cols + [c for c in FE_ALL if c in pool_fe.columns]
pp_best = fit_preprocessor(pool_fe, keep_cols=best_fe_keep)
X_pool_best = transform_X(pool_fe, pp_best)
X_tr_fi, cat_cols_fi, le_maps_fi = _prep_tree(X_pool_best)
cat_idx_fi = [list(X_tr_fi.columns).index(c) for c in cat_cols_fi]
cb_fi = CatBoostClassifier(**CB_PARAMS)
cb_fi.fit(X_tr_fi, pool_fe[TARGET].values, cat_features=cat_idx_fi, silent=True)
fi_all = pd.Series(cb_fi.feature_importances_, index=X_tr_fi.columns).sort_values(ascending=True)
# Top 30
fi_top = fi_all.tail(30)
fi_top.plot(kind='barh', ax=ax3, color=['#FF5722' if c.startswith('fe_') else '#2196F3' for c in fi_top.index])
ax3.set_title("Top 30 Feature Importance (CatBoost, Full FE)")
ax3.set_xlabel("Importance")
plt.tight_layout()
fig3_path = os.path.join(RESULTS_DIR, "fig3_feature_importance.png")
fig3.savefig(fig3_path, dpi=150)
plt.close(fig3)
print(f"Fig 3: {fig3_path}")

# Save feature importance CSV
fi_csv = fi_all.reset_index()
fi_csv.columns = ["feature", "importance"]
fi_csv.to_csv(os.path.join(RESULTS_DIR, "feature_importance.csv"), index=False)

# Fig 4: RFE Curve
if rfe_results:
    fig4, ax4 = plt.subplots(figsize=(10, 5))
    rfe_df_plot = pd.DataFrame(rfe_results).sort_values("n_fe")
    ax4.plot(rfe_df_plot["n_fe"], rfe_df_plot["boot_f1"], 'o-', color='#2196F3', linewidth=2)
    ax4.fill_between(rfe_df_plot["n_fe"],
                     rfe_df_plot["boot_f1"] - rfe_df_plot["boot_std"],
                     rfe_df_plot["boot_f1"] + rfe_df_plot["boot_std"],
                     alpha=0.2)
    ax4.axhline(y=0.7300, color='red', linestyle='--', label='NB32 Baseline')
    best_n = int(best_rfe["n_fe"])
    ax4.axvline(x=best_n, color='green', linestyle=':', label=f'Optimal ({best_n} FE)')
    ax4.set_xlabel("Number of FE Features")
    ax4.set_ylabel("Bootstrap 80/20 F1")
    ax4.set_title("Exp 3: RFE Curve")
    ax4.legend()
    plt.tight_layout()
    fig4_path = os.path.join(RESULTS_DIR, "fig4_rfe_curve.png")
    fig4.savefig(fig4_path, dpi=150)
    plt.close(fig4)
    print(f"Fig 4: {fig4_path}")

    rfe_csv = pd.DataFrame(rfe_results)
    rfe_csv.to_csv(os.path.join(RESULTS_DIR, "rfe_curve.csv"), index=False)

# Fig 5: Best Model Confusion Matrix
fig5, ax5 = plt.subplots(figsize=(6, 5))
best_r = results_df.iloc[0]
cm = np.array([[int(best_r['tn']), int(best_r['fp'])],
               [int(best_r['fn']), int(best_r['tp'])]])
im = ax5.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax5.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16,
                color='white' if cm[i,j] > cm.max()/2 else 'black')
ax5.set_xticks([0,1]); ax5.set_yticks([0,1])
ax5.set_xticklabels(['Pred 0', 'Pred 1']); ax5.set_yticklabels(['True 0', 'True 1'])
ax5.set_title(f"Best: {best_r['experiment']}\nBoot-F1={best_r['boot_f1']:.4f}")
plt.colorbar(im)
plt.tight_layout()
fig5_path = os.path.join(RESULTS_DIR, "fig5_best_confusion.png")
fig5.savefig(fig5_path, dpi=150)
plt.close(fig5)
print(f"Fig 5: {fig5_path}")


SONUC DERLEMESI
Sonuclar: /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe/advanced_fe_results.csv
Toplam deney: 37

En iyi: E1d_GroupDE_AAdelta (Boot-F1=0.7576)
NB32 Baseline: 0.7300
Fig 1: /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe/fig1_group_ablation.png
Fig 2: /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe/fig2_fe_comparison.png
Fig 3: /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe/fig3_feature_importance.png
Fig 4: /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe/fig4_rfe_curve.png
Fig 5: /Users/tefe/teknofest_model/teknofest_model/results/v18_advanced_fe/fig5_best_confusion.png


In [12]:
# Cell 12: PDF Rapor
from fpdf import FPDF
from PIL import Image

class NB33Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 6, "NB33 - KANSER Advanced Feature Engineering", align="C", new_x="LMARGIN", new_y="NEXT")
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", align="C")
    def section(self, title):
        self.set_font("Helvetica", "B", 12)
        self.cell(0, 8, title, new_x="LMARGIN", new_y="NEXT")
        self.ln(2)
    def body_text(self, txt):
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, txt)
        self.ln(2)
    def add_table(self, headers, rows, col_widths=None):
        if col_widths is None:
            col_widths = [190 // len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 8)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, str(h), border=1, align="C")
        self.ln()
        self.set_font("Helvetica", "", 7)
        for row in rows:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), border=1, align="C")
            self.ln()
        self.ln(3)
    def add_image_safe(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=20, w=w)
            self.ln(5)

pdf = NB33Report()
pdf.alias_nb_pages()
pdf.set_auto_page_break(auto=True, margin=20)

# Page 1: Title
pdf.add_page()
pdf.set_font("Helvetica", "B", 16)
pdf.cell(0, 15, "NB33: KANSER Advanced Feature Engineering", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 8, f"Tarih: {datetime.now().strftime('%Y-%m-%d %H:%M')}", align="C", new_x="LMARGIN", new_y="NEXT")
pdf.ln(10)
pdf.body_text(
    "Bu rapor, KANSER paneli icin literatur destekli gelismis feature engineering "
    "calismalarini icerir. NB32 champion recetesi (P9_REVERSE_6040 + CatBoost + 4-FE, "
    "Boot-F1=0.7300) baseline olarak alinmis, 6 feature grubu (toplam 21 feature) "
    "sistematik ablasyonla test edilmistir.\n\n"
    "Feature Gruplari:\n"
    "  A: FCS-tarzi Frekans x Konservasyon (3 feature)\n"
    "  B: EK Skor Birlesimi / Meta-Prediktor (5 feature)\n"
    "  C: Missingness / Gozlenme Sayisi (3 feature)\n"
    "  D: AA Fizikokimyasal Delta (4 feature)\n"
    "  E: AA Sinif Gecisi (2 feature)\n"
    "  F: Mevcut NB32 FE (4 feature)\n\n"
    f"Toplam deney sayisi: {len(results_df)}"
)

# Page 2: Group Ablation
pdf.add_page()
pdf.section("1. Exp 1: Feature Group Ablation")
pdf.body_text(
    "Her feature grubu NB32 baseline FE (Grup F) uzerine eklenerek test edildi. "
    "CatBoost modeli ve P9_REVERSE_6040 pool sabit tutuldu."
)
abl_rows = []
for exp_name in ablation_configs:
    r = all_results.get(exp_name)
    if r:
        delta = r["boot_f1"] - (baseline_f1 or 0)
        abl_rows.append([exp_name, f"{r['boot_f1']:.4f}", f"{r['boot_std']:.3f}",
                        f"{r['precision']:.3f}", str(r['fp']), str(r['fn']),
                        f"{delta:+.4f}"])
if abl_rows:
    pdf.add_table(["Deney", "Boot-F1", "Std", "Prec", "FP", "FN", "Delta"],
                  abl_rows, [48, 22, 18, 20, 14, 14, 22])
pdf.add_image_safe(os.path.join(RESULTS_DIR, "fig1_group_ablation.png"))

# Page 3: FE Version Comparison
pdf.add_page()
pdf.section("2. Exp 2: FE Versiyon Karsilastirmasi")
pdf.body_text(
    "Uc model tipi (CatBoost, LGBM, XGB) ile dort FE versiyonu karsilastirildi:\n"
    "  - no_fe: Ham feature'lar\n"
    "  - nb32_fe: NB32 baseline (4 feature)\n"
    "  - v2_full: add_fe_v2 tumu (21 feature)\n"
    "  - v2_safe: Grup C haric (leakage-safe, 18 feature)"
)
e2_rows = []
for mt in ["catboost", "lgbm", "xgb"]:
    for fl in ["no_fe", "nb32_fe", "v2_full", "v2_safe"]:
        k = f"E2_{mt}_{fl}"
        r = all_results.get(k)
        if r:
            e2_rows.append([f"{mt}/{fl}", f"{r['boot_f1']:.4f}", f"{r['boot_std']:.3f}",
                           f"{r['precision']:.3f}", str(r['fp']), str(r['fn'])])
if e2_rows:
    pdf.add_table(["Deney", "Boot-F1", "Std", "Prec", "FP", "FN"],
                  e2_rows, [50, 22, 18, 25, 15, 15])
pdf.add_image_safe(os.path.join(RESULTS_DIR, "fig2_fe_comparison.png"))

# Page 4: Feature Importance
pdf.add_page()
pdf.section("3. Feature Importance (CatBoost)")
pdf.add_image_safe(os.path.join(RESULTS_DIR, "fig3_feature_importance.png"))

# Page 5: RFE
pdf.add_page()
pdf.section("4. Exp 3: RFE - Optimal Feature Subset")
if rfe_results:
    pdf.body_text(
        f"Baslangic: {len(rfe_results[0]['fe_cols'])} FE feature ile RFE.\n"
        f"Optimal: {int(best_rfe['n_fe'])} FE feature, Boot-F1={best_rfe['boot_f1']:.4f}\n"
        f"Optimal featureler: {', '.join(best_rfe['fe_cols'])}"
    )
pdf.add_image_safe(os.path.join(RESULTS_DIR, "fig4_rfe_curve.png"))

# Page 6: Best Model
pdf.add_page()
pdf.section("5. En Iyi Model ve Confusion Matrix")
if len(results_df) > 0:
    best = results_df.iloc[0]
    pdf.body_text(
        f"En iyi: {best['experiment']}\n"
        f"Boot-F1: {best['boot_f1']:.4f} +/- {best['boot_std']:.3f}\n"
        f"Precision: {best['precision']:.3f}, FP={int(best['fp'])}, FN={int(best['fn'])}\n"
        f"MCC: {best['mcc']:.3f}, AUPRC: {best['auprc']:.3f}\n\n"
        f"NB32 Baseline: 0.7300\n"
        f"Delta: {best['boot_f1'] - 0.7300:+.4f}"
    )
pdf.add_image_safe(os.path.join(RESULTS_DIR, "fig5_best_confusion.png"))

# Save
pdf_path = os.path.join(REPORTS_DIR, "NB33_advanced_fe_report.pdf")
pdf.output(pdf_path)
print(f"PDF rapor: {pdf_path}")
print("NB33 tamamlandi!")

PDF rapor: /Users/tefe/teknofest_model/teknofest_model/reports/NB33_advanced_fe_report.pdf
NB33 tamamlandi!
